# Blind Spots of Qwen3.5-4B-Base

**Model**: [`Qwen/Qwen3.5-4B-Base`](https://huggingface.co/Qwen/Qwen3.5-4B-Base)
**Type**: Pre-trained base model (not instruction-tuned)
**Parameters**: ~4B
**Released**: February 2026 · **Dataset revision**: June 2026
**License**: Apache 2.0

This notebook probes the model for systematic failure modes across 12 reasoning
categories. For each category it runs **5 failure probes** (hard items) plus
**2 success controls** (easy items in the same domain). The controls give a baseline,
so we can report a *failure rate* rather than a pile of anecdotes.

It runs every prompt under **two dtypes — `float16` and `bfloat16` — and keeps both**,
because the dtype turns out to matter a lot here (see the note below).

For every output we record two things separately:
- **`output_coherent`** — did the model produce legible, on-format text at all?
- **`answer_correct`** — given a coherent output, was the answer right?

This splits failures into **format failures** (incoherent / off-format) vs
**reasoning failures** (coherent but wrong).

> **Note on dtype.** An earlier run used `float16` and produced incoherent
> "symbol-salad" output across the board — including on easy controls the model clearly
> knows. That is the classic signature of float16 numerical overflow on Qwen-class
> models, *not* a reasoning failure. We run both dtypes so the contrast is explicit: if
> `float16` is garbage even on controls while `bfloat16` is coherent, the float16
> "failures" are a numerical artifact. Prefer an **L4 / A10G** runtime (a T4 lacks fast
> bfloat16).

## 1. Setup

In [ ]:
# Drop torchvision (text-only) to avoid a torch/torchvision mismatch when loading Qwen3.5.
!pip install -q -U "transformers @ git+https://github.com/huggingface/transformers.git" \
  accelerate huggingface_hub
!pip uninstall -y -q torchvision
# Already ran later cells? Runtime > Restart session, then rerun.

In [ ]:
import gc
import json
import string
import urllib.request
from collections import Counter

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Tokenizer + Run Configuration

Qwen3.5-4B-Base uses a hybrid architecture (Gated DeltaNet linear attention + sparse
MoE + gated attention). We load the tokenizer once and run the model under each dtype
in turn (loading/freeing the weights per dtype to fit a single GPU).

In [ ]:
MODEL_NAME = "Qwen/Qwen3.5-4B-Base"
DTYPES = ["float16", "bfloat16"]   # run both and keep both, for comparison

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("Tokenizer ready. Model weights are (re)loaded per dtype in the run cell below.")

## 3. Generation + Classification Helpers

`generate` runs greedy decoding (deterministic). `classify` labels each output as:

- **`format_failure`** — incoherent / off-format (the model never produces a usable answer)
- **`reasoning_failure`** — coherent text, but the answer is wrong
- **`correct`** — coherent and right

The coherence and correctness checks are deliberately simple heuristics — a screen,
not a judge. Spot-check anything near the threshold by eye before trusting a label.

In [ ]:
def generate(model, prompt: str, max_new_tokens: int = 80) -> str:
    """Greedy completion for a given prompt (deterministic)."""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


_ALLOWED = set(string.printable)


def is_coherent(text: str) -> bool:
    """Heuristic: is the output mostly legible English / QA text?

    Returns False for 'symbol-salad' (heavy non-Latin scripts, runaway punctuation,
    repeated single tokens). A screen, not a judge.
    """
    s = text.strip()
    if not s:
        return False
    legible = sum(ch in _ALLOWED for ch in s) / len(s)
    alpha = sum(ch.isalpha() and ch.isascii() for ch in s) / len(s)
    return legible >= 0.90 and alpha >= 0.35


def is_correct(model_output: str, expected: str) -> bool:
    """Loose containment check of a normalized expected answer. Only meaningful
    when the output is coherent; always re-check near-misses by hand."""
    import re

    def norm(s):
        return re.sub("[^a-z0-9]", "", s.lower())

    out = norm(model_output)
    candidates = [expected, expected.split("-")[0].split("(")[0].strip()]
    candidates += re.findall("[0-9]+(?:\\.[0-9]+)?", expected)
    return any(norm(c) and norm(c) in out for c in candidates)


def classify(model_output: str, expected: str) -> dict:
    coherent = is_coherent(model_output)
    correct = bool(coherent and is_correct(model_output, expected))
    mode = "format_failure" if not coherent else ("correct" if correct else "reasoning_failure")
    return {"output_coherent": coherent, "answer_correct": correct, "failure_mode": mode}

## 4. Load the Prompt Set

We pull the canonical 84-prompt set from the repo (`dataset/data/prompts.jsonl`):
5 failure probes + 2 success controls for each of 12 categories — arithmetic,
character-level reasoning, cognitive reflection, spatial reasoning, formal logic,
commonsense physics, probabilistic reasoning, temporal arithmetic, factual
misconceptions, multi-hop ordering, linguistic illusions, and Winograd schemas.

In [ ]:
PROMPTS_URL = (
    "https://raw.githubusercontent.com/mohammedfirdouss/"
    "fatima-fellowship-technical-task/main/dataset/data/prompts.jsonl"
)

# Download the canonical prompt set. If you cloned the repo, you can instead point
# this at the local file: TEST_CASES = [json.loads(l) for l in open("dataset/data/prompts.jsonl")]
try:
    with urllib.request.urlopen(PROMPTS_URL) as resp:
        lines = resp.read().decode("utf-8").splitlines()
    TEST_CASES = [json.loads(l) for l in lines if l.strip()]
except Exception as e:
    print("Download failed, falling back to local prompts.jsonl:", e)
    with open("prompts.jsonl") as f:
        TEST_CASES = [json.loads(l) for l in f if l.strip()]

print(f"Loaded {len(TEST_CASES)} prompts")
print("By probe type :", dict(Counter(tc["probe_type"] for tc in TEST_CASES)))
print("Categories    :", len(set(tc["category"] for tc in TEST_CASES)))

## 5. Run All Probes Under Both dtypes

For each dtype we load the weights, run all 84 prompts, classify each output, then free
the GPU before the next dtype. Each record accumulates a `runs` entry per dtype.

In [ ]:
records = {tc["id"]: {**tc, "runs": {}} for tc in TEST_CASES}

for dtype_name in DTYPES:
    print(f"\n{'#' * 60}\n# Loading {MODEL_NAME} in {dtype_name}\n{'#' * 60}")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        dtype=getattr(torch, dtype_name),
        device_map="auto",
    )
    model.eval()

    for tc in TEST_CASES:
        output = generate(model, tc["prompt"])
        labels = classify(output, tc["expected_output"])
        records[tc["id"]]["runs"][dtype_name] = {"model_output": output, **labels}
        print(f"[{dtype_name}] [{tc['id']}] {tc['probe_type']:7s} "
              f"-> {labels['failure_mode']:18s} exp={tc['expected_output']!r}")

    # Free the GPU before the next dtype.
    del model
    gc.collect()
    torch.cuda.empty_cache()

results = [records[tc["id"]] for tc in TEST_CASES]
print(f"\nDone: {len(results)} prompts x {len(DTYPES)} dtypes.")

## 6. Failure-Rate Summary (per dtype) and Save

The headline result: per dtype, the failure rate on hard probes vs the success rate on
easy controls, with failures split into format vs reasoning. The float16-vs-bfloat16
contrast on the *controls* is what reveals whether the incoherence is a numerical
artifact. Full records (with both runs) are saved to JSONL afterwards.

In [ ]:
def rate(rows, pred):
    if not rows:
        return "n/a"
    n = sum(1 for r in rows if pred(r))
    return f"{n}/{len(rows)} ({100 * n / len(rows):.0f}%)"

failures = [r for r in results if r["probe_type"] == "failure"]
controls = [r for r in results if r["probe_type"] == "control"]

print("=" * 64)
print("SUMMARY (per dtype)")
print("=" * 64)
print(f"Failure probes: {len(failures)}   Success controls: {len(controls)}")

for dt in DTYPES:
    mode = lambda r: r["runs"][dt]["failure_mode"]
    coherent = lambda r: r["runs"][dt]["output_coherent"]
    print(f"\n--- {dt} ---")
    print("  failure probes  coherent          :", rate(failures, coherent))
    print("  failure probes  correct           :", rate(failures, lambda r: mode(r) == "correct"))
    print("    format_failure                  :", rate(failures, lambda r: mode(r) == "format_failure"))
    print("    reasoning_failure               :", rate(failures, lambda r: mode(r) == "reasoning_failure"))
    print("  controls        coherent          :", rate(controls, coherent))
    print("  controls        correct           :", rate(controls, lambda r: mode(r) == "correct"))

print("\nRead: if float16 is incoherent even on the easy CONTROLS while bfloat16 is")
print("coherent on them, the float16 'failures' are a numerical artifact, not a blind spot.")

OUTPUT_FILE = "blind_spots_data.jsonl"
with open(OUTPUT_FILE, "w") as f:
    for r in results:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
print(f"\nSaved {len(results)} records to {OUTPUT_FILE}")
print("Spot-check the auto labels, then use this file to update dataset/data/train.jsonl.")

## 7. (Optional) Upload Dataset to Hugging Face

After spot-checking the labels and updating the dataset README, you can push directly
from Colab. The uploaded `train.jsonl` should be the spot-checked version.

In [ ]:
from huggingface_hub import HfApi, login

login()
api = HfApi()
api.create_repo("mohammedfirdouss/qwen35-4b-base-blind-spots", repo_type="dataset", exist_ok=True)
api.upload_folder(
    folder_path="dataset/",
    repo_id="mohammedfirdouss/qwen35-4b-base-blind-spots",
    repo_type="dataset",
)